In [ ]:
from dotenv import load_dotenv
import os,time
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_ollama import ChatOllama
from uuid import uuid4
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore
from langchain_core.documents import Document


# Load the .env file
load_dotenv()

pinecone_api_key = os.getenv("PINECONE_API_KEY")


#----------------------------------------------------
# 1. Embedding model
#----------------------------------------------------
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")



#----------------------------------------------------
# 2. Chat model
#----------------------------------------------------
model = ChatOllama(model="qwen3")



# ---------------------------------------------------
# 3. Pinecone client
# ---------------------------------------------------
pc = Pinecone(
    api_key=pinecone_api_key
)


# ---------------------------------------------------
# 4. Create index
# ---------------------------------------------------
index_name = "langchain-llama-index"
dimension = 384
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=dimension,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        ),
    )


# ---------------------------------------------------
# 5. Wait for index readiness
# ---------------------------------------------------
while True:
    description = pc.describe_index(index_name)
    if description.status["ready"]:
        break
    time.sleep(2)


# ---------------------------------------------------
# 6. Connect to index
# ---------------------------------------------------
index = pc.Index(index_name)


# ---------------------------------------------------
# 7. LangChain Pinecone vector store
# ---------------------------------------------------

vector_store = PineconeVectorStore(
    index=index,
    embedding=embeddings,
    namespace="demo-documents"
)


# ---------------------------------------------------
# 8. Create documents
# ---------------------------------------------------

documents = [
    Document(
        page_content=(
            "I had chocolate chip pancakes and "
            "scrambled eggs for breakfast this morning."
        ),
        metadata={"source": "tweet"},
    ),
    Document(
        page_content=(
            "The weather forecast for tomorrow is cloudy "
            "and overcast, with a high of 62 degrees."
        ),
        metadata={"source": "news"},
    ),
    Document(
        page_content=(
            "Building an exciting new project with "
            "LangChain - come check it out!"
        ),
        metadata={"source": "tweet"},
    ),
    Document(
        page_content=(
            "Robbers broke into the city bank and "
            "stole $1 million in cash."
        ),
        metadata={"source": "news"},
    ),
    Document(
        page_content=(
            "LangGraph is the best framework for building "
            "stateful, agentic applications!"
        ),
        metadata={"source": "tweet"},
    ),
]

ids = [
    str(uuid4())
    for _ in documents
]


# ---------------------------------------------------
# 9. Add documents
# ---------------------------------------------------

inserted_ids = vector_store.add_documents(
    documents=documents,
    ids=ids
)

print("Inserted IDs:", inserted_ids)


# ---------------------------------------------------
# 10. Similarity search with metadata filter
# ---------------------------------------------------

results = vector_store.similarity_search(
    query=("LangChain provides abstractions for working with LLMs"),
    k=2,
    filter={"source": "tweet"}
)

for result in results:
    print(result.page_content)
    print(result.metadata)


# ---------------------------------------------------
# 11. Retriever with threshold and filter
# ---------------------------------------------------

retriever = vector_store.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={
        "k": 1,
        "score_threshold": 0.4,
        "filter": {
            "source": "news"
        }
    }
)

retrieved_docs = retriever.invoke(
    "Stealing money from a bank is a crime"
)

print(retrieved_docs)


d:\RAG\rag_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2452.57it/s]


Inserted IDs: ['42e6c9e5-f9f3-49de-b556-72921bf1d375', 'adb4467f-f324-40c6-8d05-81e980d75e31', 'acaeb758-8524-4733-a352-ff4cad89be62', '9f977f96-a61e-43e4-8854-bc7dd2dfde61', '3c571e4c-d7a4-485e-a0e2-52e1ae5ee1ad']
LangGraph is the best framework for building stateful, agentic applications!
{'source': 'tweet'}
Building an exciting new project with LangChain - come check it out!
{'source': 'tweet'}
[Document(id='9f977f96-a61e-43e4-8854-bc7dd2dfde61', metadata={'source': 'news'}, page_content='Robbers broke into the city bank and stole $1 million in cash.')]
